# Final Model, Threshold Tuning, & Explainability
In this notebook, we finalize our Random Forest model. We find the mathematically optimal decision threshold on the training data, test its performance, explain the model's inner workings using SHAP, and save the model for production.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, confusion_matrix, precision_score, recall_score, f1_score

# 1. Load Data & Split
df = pd.read_csv('../data/processed.csv')
X = df.drop(columns=['isFraud', 'step'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data Split Complete. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 1. Choosing the Threshold
**Why do we choose the threshold on the training data?**
Machine learning models output a probability (from 0% to 100%) of a transaction being fraud. By default, anything over 50% is flagged as fraud. However, for highly imbalanced problems, 50% is rarely the best threshold. We might want to lower the bar to 30% to catch more fraud, or raise it to 70% to reduce false alarms.

We *must* choose this threshold using only the training data (via cross-validation). If we look at the test data to pick the best threshold, we are "cheating." The test data is our final exam; we can't use it to study. By finding the best threshold on the training data, we ensure our model's performance on the test data is a realistic representation of how it will behave in the real world.

In [ ]:
# Setup Cross-Validation for Threshold tuning
rf_cv = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Calculating cross-validated probabilities on the training set...")
y_train_probs = cross_val_predict(rf_cv, X_train, y_train, cv=cv, method='predict_proba', n_jobs=2)[:, 1]

# Calculate precision, recall, and F1 for all possible thresholds
precisions, recalls, thresholds = precision_recall_curve(y_train, y_train_probs)
# Avoid division by zero warning
f1_scores = np.divide(2 * (precisions[:-1] * recalls[:-1]), (precisions[:-1] + recalls[:-1]), out=np.zeros_like(precisions[:-1]), where=(precisions[:-1] + recalls[:-1])!=0)

# Find the threshold that gives the absolute best F1 Score
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Best Threshold chosen: {best_threshold:.4f} (F1 Score: {f1_scores[best_idx]:.4f})")

# Plot the curve
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions[:-1], 'b--', label='Precision')
plt.plot(thresholds, recalls[:-1], 'g-', label='Recall')
plt.axvline(x=best_threshold, color='r', linestyle=':', label=f'Best Threshold ({best_threshold:.4f})')
plt.legend(loc='center left')
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision and Recall vs. Threshold (Training Data)")
plt.grid(True, alpha=0.3)
plt.show()

## 2. Train Final Model & Compare Thresholds on Test Set
Now we train the final model on 100% of the training data and evaluate it on the untouched test data to see if our custom threshold actually helped.

In [ ]:
print("Training final Random Forest on full training set...")
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_test_probs = rf.predict_proba(X_test)[:, 1]

# Make predictions using the Default 0.5 threshold
y_pred_default = (y_test_probs >= 0.5).astype(int)

# Make predictions using our Optimized threshold
y_pred_best = (y_test_probs >= best_threshold).astype(int)

def print_metrics(y_true, y_pred, title):
    print(f"\n--- {title} ---")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")

print_metrics(y_test, y_pred_default, "Metrics at Default Threshold (0.5000)")
print_metrics(y_test, y_pred_best, f"Metrics at Optimized Threshold ({best_threshold:.4f})")

## 3. Explaining the Model with SHAP
SHAP (SHapley Additive exPlanations) is a game-theoretic approach to explain the output of any machine learning model. It tells us exactly *why* a model made a specific decision.

**How to read the Beeswarm plot:**
- Every dot is a single transaction.
- **Position on the X-axis:** Tells you if the feature pushed the model toward guessing "Fraud" (right side, positive SHAP value) or "Not Fraud" (left side, negative SHAP value).
- **Color:** Tells you the actual value of the feature for that transaction. Red means the feature was high (e.g., a huge amount of money). Blue means the feature was low.
- *Example:* If you see lots of red dots on the far right for `errorBalanceOrig`, it means high balance errors heavily pushed the model into screaming "Fraud!"

**How to read the Waterfall plot:**
- This explains *one single* fraudulent transaction.
- It starts at the bottom with the model's "Base value" (what it would guess if it had no information).
- Each bar shows how each specific feature pushed the probability up (red) or down (blue) until it reaches the final prediction at the top.

In [ ]:
# Get all fraud cases in the test set
fraud_idx = y_test[y_test == 1].index
# Sample 400 normal cases
normal_idx = y_test[y_test == 0].sample(n=400, random_state=42).index

sample_idx = fraud_idx.union(normal_idx)
X_shap = X_test.loc[sample_idx]

print("Calculating SHAP values (this takes a moment)...")
explainer = shap.TreeExplainer(rf)
shap_values = explainer(X_shap)

# Note: In newer SHAP versions with TreeExplainer on RandomForest, 
# shap_values represents both classes (shape = [samples, features, classes]).
# We slice [:, :, 1] to only plot the explanations for the Fraud (1) class.

print("\n--- Global Explanation: SHAP Beeswarm Plot ---")
shap.plots.beeswarm(shap_values[:, :, 1])

print("\n--- Local Explanation: SHAP Waterfall Plot for ONE Fraudulent Transaction ---")
# Find the position of the first fraud case in our sample
fraud_loc = np.where(y_test.loc[sample_idx] == 1)[0][0]
shap.plots.waterfall(shap_values[fraud_loc, :, 1])

## 4. Save Model and Metadata
We save the finalized model, the optimal threshold, and the feature names so our future API (FastAPI) can load them and make real-time predictions.

In [ ]:
# Save the model to a file
joblib.dump(rf, '../models/fraud_model.joblib')

# Save the optimal threshold and exact list of features
meta = {
    'threshold': float(best_threshold), 
    'features': list(X_train.columns)
}
with open('../models/model_meta.json', 'w') as f:
    json.dump(meta, f)

print("Model and metadata successfully saved to the models/ directory!")